# Senate Money x Polymarket Overview

This notebook takes the current 2026 Senate finance and Polymarket files and asks a few quick questions:

- Are Polymarket favorites also the better-funded side?
- Are closer races attracting more money?
- Which races look expensive, close, or a bit out of sync between money and odds?

Important caveat:
The raw `senate_money_snapshot_2026.csv` currently contains many stale coverage dates from older cycles. So below we rebuild a cleaner current-cycle snapshot by filtering the finance totals to `coverage_end_date >= 2025-01-01` and then taking the top Democrat and Republican by receipts in each state.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

base = Path('..') if Path('..', 'data').exists() else Path('.')
processed = base / 'data' / 'processed'
analysis_dir = base / 'analysis'
analysis_dir.mkdir(exist_ok=True)

raw_snapshot = pd.read_csv(processed / 'senate_money_snapshot_2026.csv')
polymarket = pd.read_csv(processed / 'polymarket_senate_top_two_2026.csv')
totals = pd.read_csv(processed / 'senate_candidate_finance_totals_2026.csv')

print('raw snapshot rows:', len(raw_snapshot))
print('polymarket states:', polymarket['state'].nunique())
print('totals rows:', len(totals))

## 1. Sanity-check the raw snapshot

In [ ]:
for prefix in ['dem', 'rep']:
    raw_snapshot[f'{prefix}_coverage_end_date'] = pd.to_datetime(raw_snapshot[f'{prefix}_coverage_end_date'], errors='coerce')

pd.Series({
    'dem rows with coverage_end_date before 2025-01-01': int((raw_snapshot['dem_coverage_end_date'] < pd.Timestamp('2025-01-01')).sum()),
    'rep rows with coverage_end_date before 2025-01-01': int((raw_snapshot['rep_coverage_end_date'] < pd.Timestamp('2025-01-01')).sum()),
    'total states in raw snapshot': len(raw_snapshot),
})

## 2. What is in the totals file by cycle and coverage year?

In [ ]:
totals['coverage_end_date'] = pd.to_datetime(totals['coverage_end_date'], errors='coerce')
totals['cycle'] = pd.to_numeric(totals['cycle'], errors='coerce')

cycle_counts = totals['cycle'].value_counts(dropna=False).sort_index()
coverage_year_counts = totals['coverage_end_date'].dt.year.value_counts(dropna=False).sort_index()

print('cycle counts')
display(cycle_counts.to_frame('rows'))
print('coverage end year counts')
display(coverage_year_counts.to_frame('rows'))

## 3. Build a cleaner current-cycle race snapshot

In [ ]:
for col in ['total_receipts', 'total_disbursements', 'cash_on_hand', 'debts_owed_by_committee']:
    totals[col] = pd.to_numeric(totals.get(col), errors='coerce').fillna(0)

current = totals[totals['coverage_end_date'] >= pd.Timestamp('2025-01-01')].copy()
current = current.sort_values(['fec_candidate_id', 'coverage_end_date', 'total_receipts'], ascending=[True, False, False])
current_latest = current.drop_duplicates(subset=['fec_candidate_id'], keep='first').copy()
current_latest = current_latest[current_latest['party'].isin(['DEM', 'REP'])]

selected = current_latest.sort_values(['state', 'party', 'total_receipts'], ascending=[True, True, False])
selected = selected.drop_duplicates(['state', 'party'])
selected = selected[['state', 'party', 'fec_candidate_id', 'candidate_name', 'total_receipts', 'total_disbursements', 'cash_on_hand', 'debts_owed_by_committee', 'coverage_end_date']]

pivot = selected.pivot(index='state', columns='party')
pivot.columns = [f'{party.lower()}_{col}' for col, party in pivot.columns]
pivot = pivot.reset_index()

print('states with both selected sides:', len(pivot))
pivot.head()

## 4. Join to Polymarket top-two data

In [ ]:
pm = polymarket[['state', 'top_1_entity_name', 'top_1_entity_type', 'top_1_yes_probability', 'top_1_market_slug', 'top_1_fec_candidate_id', 'top_2_entity_name', 'top_2_entity_type', 'top_2_yes_probability', 'top_2_market_slug', 'top_2_fec_candidate_id']].copy()
for col in ['top_1_yes_probability', 'top_2_yes_probability']:
    pm[col] = pd.to_numeric(pm[col], errors='coerce')

analysis = pivot.merge(pm, on='state', how='inner')
analysis['favorite_margin_prob'] = (analysis['top_1_yes_probability'] - analysis['top_2_yes_probability']).abs()
analysis['combined_receipts'] = analysis['dem_total_receipts'].fillna(0) + analysis['rep_total_receipts'].fillna(0)
analysis['combined_spend'] = analysis['dem_total_disbursements'].fillna(0) + analysis['rep_total_disbursements'].fillna(0)
analysis['combined_cash'] = analysis['dem_cash_on_hand'].fillna(0) + analysis['rep_cash_on_hand'].fillna(0)
analysis['dem_receipts_share'] = analysis['dem_total_receipts'] / analysis['combined_receipts'].replace(0, np.nan)
analysis['rep_receipts_share'] = analysis['rep_total_receipts'] / analysis['combined_receipts'].replace(0, np.nan)
analysis['dem_cash_share'] = analysis['dem_cash_on_hand'] / analysis['combined_cash'].replace(0, np.nan)
analysis['rep_cash_share'] = analysis['rep_cash_on_hand'] / analysis['combined_cash'].replace(0, np.nan)

analysis['favorite_party'] = pd.Series(pd.NA, index=analysis.index, dtype='object')
analysis.loc[analysis['top_1_entity_type'].eq('party') & analysis['top_1_entity_name'].str.contains('Democrat', case=False, na=False), 'favorite_party'] = 'DEM'
analysis.loc[analysis['top_1_entity_type'].eq('party') & analysis['top_1_entity_name'].str.contains('Republican', case=False, na=False), 'favorite_party'] = 'REP'
analysis.loc[analysis['favorite_party'].isna() & (analysis['top_1_fec_candidate_id'] == analysis['dem_fec_candidate_id']), 'favorite_party'] = 'DEM'
analysis.loc[analysis['favorite_party'].isna() & (analysis['top_1_fec_candidate_id'] == analysis['rep_fec_candidate_id']), 'favorite_party'] = 'REP'

analysis['favorite_receipts_share'] = np.where(analysis['favorite_party'] == 'DEM', analysis['dem_receipts_share'], np.where(analysis['favorite_party'] == 'REP', analysis['rep_receipts_share'], np.nan))
analysis['favorite_cash_share'] = np.where(analysis['favorite_party'] == 'DEM', analysis['dem_cash_share'], np.where(analysis['favorite_party'] == 'REP', analysis['rep_cash_share'], np.nan))
analysis['favorite_candidate_name'] = pd.Series(pd.NA, index=analysis.index, dtype='object')
analysis.loc[analysis['favorite_party'] == 'DEM', 'favorite_candidate_name'] = analysis.loc[analysis['favorite_party'] == 'DEM', 'dem_candidate_name']
analysis.loc[analysis['favorite_party'] == 'REP', 'favorite_candidate_name'] = analysis.loc[analysis['favorite_party'] == 'REP', 'rep_candidate_name']
analysis['favorite_receipts_gap_vs_opponent'] = np.where(analysis['favorite_party'] == 'DEM', analysis['dem_total_receipts'] - analysis['rep_total_receipts'], np.where(analysis['favorite_party'] == 'REP', analysis['rep_total_receipts'] - analysis['dem_total_receipts'], np.nan))
analysis['favorite_cash_gap_vs_opponent'] = np.where(analysis['favorite_party'] == 'DEM', analysis['dem_cash_on_hand'] - analysis['rep_cash_on_hand'], np.where(analysis['favorite_party'] == 'REP', analysis['rep_cash_on_hand'] - analysis['dem_cash_on_hand'], np.nan))

print('states in joined analysis:', len(analysis))
analysis[['state', 'dem_candidate_name', 'rep_candidate_name', 'top_1_entity_name', 'top_1_yes_probability', 'top_2_entity_name', 'top_2_yes_probability']].head()

## 5. Correlations

In [ ]:
corr_cols = ['top_1_yes_probability', 'favorite_margin_prob', 'combined_receipts', 'combined_spend', 'combined_cash', 'favorite_receipts_share', 'favorite_cash_share']
correlations = analysis[corr_cols].corr(numeric_only=True)
correlations.loc[['top_1_yes_probability', 'favorite_margin_prob'], ['combined_receipts', 'combined_spend', 'combined_cash', 'favorite_receipts_share', 'favorite_cash_share']]

## 6. Closest races by Polymarket

In [ ]:
closest = analysis.sort_values('favorite_margin_prob')[['state', 'top_1_entity_name', 'top_1_yes_probability', 'top_2_entity_name', 'top_2_yes_probability', 'dem_candidate_name', 'rep_candidate_name', 'combined_receipts', 'combined_spend', 'combined_cash']].head(12)
closest

## 7. Most expensive races

In [ ]:
most_expensive = analysis.sort_values('combined_receipts', ascending=False)[['state', 'dem_candidate_name', 'rep_candidate_name', 'combined_receipts', 'combined_spend', 'combined_cash', 'favorite_margin_prob', 'top_1_entity_name', 'top_1_yes_probability']].head(12)
most_expensive

## 8. Places where the market favorite does not have most of the receipts

In [ ]:
money_market_tension = analysis[analysis['favorite_receipts_share'] < 0.5][['state', 'top_1_entity_name', 'favorite_party', 'favorite_candidate_name', 'favorite_receipts_share', 'favorite_cash_share', 'dem_candidate_name', 'rep_candidate_name', 'dem_total_receipts', 'rep_total_receipts']].sort_values('favorite_receipts_share')
money_market_tension

## 9. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(analysis['favorite_receipts_share'], analysis['top_1_yes_probability'])
axes[0].set_title('Favorite Receipts Share vs Favorite Win Probability')
axes[0].set_xlabel('Favorite share of combined receipts')
axes[0].set_ylabel('Favorite Polymarket probability')
axes[0].grid(alpha=0.3)

axes[1].scatter(analysis['favorite_margin_prob'], analysis['combined_receipts'])
axes[1].set_title('Tighter Races vs Combined Receipts')
axes[1].set_xlabel('Favorite probability margin')
axes[1].set_ylabel('Combined receipts')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
closest_plot = analysis.sort_values('favorite_margin_prob').head(10).copy()
closest_plot = closest_plot.sort_values('favorite_margin_prob', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(closest_plot['state'], closest_plot['favorite_margin_prob'])
plt.title('Closest Senate Races by Polymarket Margin')
plt.xlabel('Favorite probability margin')
plt.ylabel('State')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.show()

## 10. Save the analysis table

In [ ]:
output_path = analysis_dir / 'senate_money_polymarket_current_cycle_analysis.csv'
analysis.to_csv(output_path, index=False)
output_path

## Preliminary read

A few early takeaways from this version of the data:

- Total money in a race does **not** seem to line up strongly with a race being safer for one side. If anything, closer races often look more expensive.
- The market favorite's **share** of the money looks more informative than the total amount of money in the race.
- There are a few interesting tension states where the market favorite is not the side with most receipts, which could be worth a manual check.
- The raw snapshot should probably not be used directly for serious analysis until the current-cycle filtering is built into the production pipeline.
